# EVE Voice Engine — تدريب نموذج ليبي صغير
هذا دفتر تجريبي للتدريب السحابي. يستخدم نموذجاً معلماً لإنتاج نصوص أولية فقط، ثم يدرب طالب CTC صغيراً بأوزان عشوائية. لا توجد أوزان EVE مدربة مسبقاً في هذا المستودع.

**تنبيه:** لا تستخدم بيانات صوتية إلا إذا كان لديك حق استخدامها للتدريب. يجب أن تكون مجموعة التحقق مكتوبة يدوياً، وألا تدخل بياناتها في التدريب.

In [ ]:
!pip -q install -r https://raw.githubusercontent.com/khyralslh44-hub/WALLE-Offline-New/eve-voice-engine-distillation/eve-voice-engine/requirements.txt
!pip -q install jiwer

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, json, re, math, shutil, subprocess
import numpy as np, pandas as pd, torch, soundfile as sf, librosa
from datasets import Dataset
from transformers import (AutoProcessor, pipeline, Wav2Vec2Config, Wav2Vec2ForCTC,
    Wav2Vec2CTCTokenizer, Wav2Vec2FeatureExtractor, Wav2Vec2Processor,
    TrainingArguments, Trainer)
from transformers import DataCollatorCTCWithPadding
from jiwer import wer, cer

ROOT = Path('/content/drive/MyDrive/EVE_Libyan_ASR')
TRAIN_CSV = ROOT / 'train.csv'
VALID_CSV = ROOT / 'valid.csv'
assert TRAIN_CSV.exists() and VALID_CSV.exists(), 'ضع train.csv و valid.csv داخل مجلد EVE_Libyan_ASR في Google Drive'
train_df = pd.read_csv(TRAIN_CSV).fillna('')
valid_df = pd.read_csv(VALID_CSV).fillna('')
assert {'audio_path','transcript'}.issubset(train_df.columns)
assert {'audio_path','transcript'}.issubset(valid_df.columns)
assert (valid_df.transcript.astype(str).str.strip() != '').all(), 'كل نصوص valid.csv يجب أن تكون مكتوبة ومراجعة بشرياً'
print('Train rows:', len(train_df), 'Valid rows:', len(valid_df), 'GPU:', torch.cuda.is_available())
if 'speaker_id' in train_df.columns and 'speaker_id' in valid_df.columns:
    tr=set(train_df.speaker_id.astype(str)) - {''}
    va=set(valid_df.speaker_id.astype(str)) - {''}
    overlap=tr & va
    assert not overlap, f'تداخل متحدثين بين train وvalid: {len(overlap)}؛ افصل المتحدثين لتقييم عادل'
    print('Speaker-disjoint split verified')


## 1) المعلم: إنتاج تسميات أولية للمقاطع التي لا تملك نصاً
المعلم يُستخدم في السحابة فقط. راجع وصحح النصوص الناتجة؛ لا تعتبرها حقيقة مؤكدة.

In [ ]:
TEACHER_ID = 'Tass02/whisper-small-libyan'
teacher = pipeline('automatic-speech-recognition', model=TEACHER_ID,
                   device=0 if torch.cuda.is_available() else -1,
                   chunk_length_s=25)

def resolve_audio(p):
    p = Path(str(p))
    return p if p.is_absolute() else ROOT / p

def get_teacher_text(path):
    result = teacher(str(resolve_audio(path)))
    return str(result.get('text','')).strip()

for i, row in train_df.iterrows():
    if not str(row['transcript']).strip():
        train_df.at[i, 'transcript'] = get_teacher_text(row['audio_path'])
    if i % 20 == 0:
        print('processed', i, '/', len(train_df))

train_df = train_df[train_df.transcript.astype(str).str.strip() != ''].copy()
train_df.to_csv(ROOT / 'train_pseudolabeled_review.csv', index=False)
print('Saved:', ROOT / 'train_pseudolabeled_review.csv')
print(train_df[['audio_path','transcript']].head(10).to_string(index=False))

## 2) مراجعة التسميات
افتح `train_pseudolabeled_review.csv` في Drive وصحح الأخطاء، ثم احفظ النسخة المصححة باسم `train_final.csv`. لا تبدأ التدريب النهائي قبل المراجعة.

In [ ]:
FINAL_TRAIN = ROOT / 'train_final.csv'
assert FINAL_TRAIN.exists(), 'راجع train_pseudolabeled_review.csv واحفظ النسخة المصححة باسم train_final.csv'
train_df = pd.read_csv(FINAL_TRAIN).fillna('')
train_df = train_df[train_df.transcript.astype(str).str.strip() != ''].copy()
assert len(train_df) > 0
print('Approved training rows:', len(train_df))

## 3) بناء مفردات الحروف العربية وتجهيز الصوت 16 kHz
النموذج الطالب يستخدم مفردات محارف، لتقليل حجم المفردات وتعقيد فك الترميز.

In [ ]:
def normalize_text(s):
    s = str(s).replace('ـ','')
    s = re.sub(r'[ًٌٍَُِّْٰ]', '', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

all_text = ''.join(normalize_text(x) for x in train_df.transcript.tolist())
vocab = sorted(set(all_text) - {' '})
vocab_dict = {ch:i for i,ch in enumerate(vocab)}
vocab_dict['|'] = len(vocab_dict)
vocab_dict['[UNK]'] = len(vocab_dict)
vocab_dict['[PAD]'] = len(vocab_dict)
OUT = ROOT / 'eve_student_output'
OUT.mkdir(parents=True, exist_ok=True)
with open(OUT/'vocab.json','w',encoding='utf-8') as f: json.dump(vocab_dict,f,ensure_ascii=False,indent=2)
tokenizer = Wav2Vec2CTCTokenizer(str(OUT/'vocab.json'), unk_token='[UNK]', pad_token='[PAD]', word_delimiter_token='|')
feature_extractor = Wav2Vec2FeatureExtractor(feature_size=1, sampling_rate=16000, padding_value=0.0, do_normalize=True, return_attention_mask=True)
processor = Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)

def load_wave(path):
    audio, sr = sf.read(str(resolve_audio(path)))
    if audio.ndim > 1: audio = audio.mean(axis=1)
    if sr != 16000: audio = librosa.resample(audio.astype(np.float32), orig_sr=sr, target_sr=16000)
    return audio.astype(np.float32)

def encode_rows(frame):
    rows=[]
    for r in frame.itertuples(index=False):
        audio=load_wave(r.audio_path)
        text=normalize_text(r.transcript)
        rows.append({'input_values':processor(audio, sampling_rate=16000).input_values[0],
                     'labels':tokenizer(text.replace(' ','|')).input_ids})
    return Dataset.from_list(rows)

train_ds=encode_rows(train_df)
valid_ds=encode_rows(valid_df)
print('Encoded:', len(train_ds), len(valid_ds), 'vocab:', len(vocab_dict))

## 4) إنشاء الطالب الصغير من أوزان عشوائية
هذه ليست نسخة Whisper مصغرة. إنها بنية صوتية CTC جديدة تتعلم من التسميات الليبية المصححة.

In [ ]:
config = Wav2Vec2Config(
    vocab_size=len(vocab_dict), hidden_size=256, num_hidden_layers=6,
    num_attention_heads=4, intermediate_size=1024,
    conv_dim=(64,64,128,128,256,256,256),
    conv_stride=(5,2,2,2,2,2,2), conv_kernel=(10,3,3,3,3,2,2),
    hidden_dropout=0.1, attention_dropout=0.1, feat_proj_dropout=0.1,
    feat_extract_norm='layer', ctc_loss_reduction='mean', pad_token_id=tokenizer.pad_token_id,
    bos_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.pad_token_id)
model = Wav2Vec2ForCTC(config)
print('Student parameters:', sum(p.numel() for p in model.parameters()))

collator = DataCollatorCTCWithPadding(processor=processor, padding=True)
def metrics(pred):
    logits=pred.predictions
    pred_ids=np.argmax(logits,axis=-1)
    label_ids=pred.label_ids.copy()
    label_ids[label_ids == -100]=tokenizer.pad_token_id
    hyp=processor.batch_decode(pred_ids)
    ref=processor.batch_decode(label_ids, group_tokens=False)
    return {'wer':wer(ref,hyp), 'cer':cer(ref,hyp)}

args=TrainingArguments(
    output_dir=str(ROOT/'eve_student_checkpoints'),
    per_device_train_batch_size=4, per_device_eval_batch_size=4,
    gradient_accumulation_steps=4, learning_rate=3e-4, warmup_steps=200,
    num_train_epochs=25, evaluation_strategy='epoch', save_strategy='epoch',
    logging_steps=25, fp16=torch.cuda.is_available(),
    load_best_model_at_end=True, metric_for_best_model='wer', greater_is_better=False,
    save_total_limit=2, report_to='none')
trainer=Trainer(model=model,args=args,train_dataset=train_ds,eval_dataset=valid_ds,
                tokenizer=processor,data_collator=collator,compute_metrics=metrics)
trainer.train()
print(trainer.evaluate())
trainer.save_model(str(OUT))
processor.save_pretrained(str(OUT))

## 5) قياس الحجم وتصدير ONNX وتكميمه
هذه خطوة تصدير أولية؛ يجب اختبار النموذج الناتج على Android قبل دمجه في التطبيق.

In [ ]:
# تثبيت Optimum للتصدير
!pip -q install 'optimum[onnxruntime]'
!optimum-cli export onnx --model /content/drive/MyDrive/EVE_Libyan_ASR/eve_student_output /content/drive/MyDrive/EVE_Libyan_ASR/eve_student_onnx

from onnxruntime.quantization import quantize_dynamic, QuantType
onnx_dir=ROOT/'eve_student_onnx'
for f in onnx_dir.glob('*.onnx'):
    q=f.with_name(f.stem+'_int8.onnx')
    quantize_dynamic(str(f),str(q),weight_type=QuantType.QInt8)
    print(f.name, f.stat().st_size/1024/1024, 'MB')
    print(q.name, q.stat().st_size/1024/1024, 'MB')

print('Student output files:')
for f in OUT.rglob('*'):
    if f.is_file(): print(f.relative_to(ROOT), round(f.stat().st_size/1024/1024,2),'MB')

## 6) بوابة القبول قبل دمجه في EVE
لا تدمج النموذج لمجرد نجاح التدريب. احتفظ بمجموعة اختبار منفصلة من متحدثين لم يدخلوا التدريب، وقِس WER/CER، ثم اختبر سرعة الاستدلال والذاكرة والحرارة على Redmi 12. لا تتوفر في هذه المرحلة أوزان نهائية ولا نتائج هاتف فعلية.